# 02: Advanced SQL Feature Engineering

### [ENVIRONMENT CHECK]
Run this cell to verify your kernel and install missing dependencies.

In [1]:
import sys
import os
print(f"Kernel Python Executable: {sys.executable}")
print(f"Python Version: {sys.version}")

# Auto-install missing libraries in THIS specific kernel
!{sys.executable} -m pip install -q pandas numpy duckdb

Kernel Python Executable: c:\Users\vaibh\AppData\Local\Programs\Python\Python313\python.exe
Python Version: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Objective:** Engineer high-dimensional temporal features using SQL Window Functions via DuckDB.

In [2]:
# [STAGE 1]: SQL Environment & DuckDB Initialization
import duckdb
import pandas as pd
import os

path = '../data/spotify_churn_dataset.csv' if os.path.exists('../data/spotify_churn_dataset.csv') else 'data/spotify_churn_dataset.csv'
df_raw = pd.read_csv(path)
df_raw.columns = df_raw.columns.str.strip()

In [3]:
# [STAGE 2]: SQL Window Functions: Engineered Behavioral Benchmarks
# Calculating cohort-relative listening intensity and segment-based skip rankings
con = duckdb.connect(database=':memory:')
con.register('raw_data', df_raw)

engineering_query = """
SELECT 
    *, 
    -- Feature 1: Listening intensity percentile rank within the entire population
    PERCENT_RANK() OVER (ORDER BY avg_daily_minutes) as intensity_percentile,
    
    -- Feature 2: Negative feedback density score (Skips vs Minutes)
    (skips_per_day * 1.0 / NULLIF(avg_daily_minutes, 0)) as negative_feedback_density,
    
    -- Feature 3: Support interaction load (Relative to usage frequency)
    (support_tickets * 1.0 / NULLIF(avg_daily_minutes, 0)) as support_load_score,
    
    -- Feature 4: Within-Segment Attrition Risk (Ranking skips by subscription type)
    RANK() OVER (PARTITION BY subscription_type ORDER BY skips_per_day DESC) as segment_skip_rank,
    
    -- Feature 5: Rolling 7-Day Engagement Proxy (Capturing activity density trends)
    AVG(avg_daily_minutes) OVER (
        ORDER BY days_since_last_login 
        ROWS BETWEEN 7 PRECEDING AND CURRENT ROW
    ) as rolling_7d_intensity_avg
FROM raw_data
"""

df_engineered = con.execute(engineering_query).df()
df_engineered.head()

,user_id,subscription_type,country,avg_daily_minutes,number_of_playlists,top_genre,skips_per_day,support_tickets,days_since_last_login,churned,intensity_percentile,negative_feedback_density,support_load_score,segment_skip_rank,rolling_7d_intensity_avg
0,user_99,Premium,BR,184.6,3,Rock,7,1,0,0,0.986987,0.037920,0.005417,92,184.600000
1,user_87,Premium,PK,138.7,6,Rock,4,0,0,0,0.828829,0.028839,0.000000,314,161.650000
2,user_870,Premium,FR,90.5,3,Rock,5,0,0,0,0.434434,0.055249,0.000000,238,137.933333
3,user_875,Premium,FR,135.1,8,Pop,7,0,0,0,0.796797,0.051813,0.000000,92,137.225000
4,user_573,Premium,DE,53.6,8,Country,0,1,0,1,0.173173,0.000000,0.018657,575,120.500000


In [4]:
# [STAGE 3]: SQL Window Validation: Temporal Engagement Sanity Check
# Confirming the rolling average windows were correctly calculated in the main set
print("Validation of Integrated Features:")
print(df_engineered[['user_id', 'days_since_last_login', 'avg_daily_minutes', 'rolling_7d_intensity_avg']].head(10))

Validation of Integrated Features:
    user_id  days_since_last_login  avg_daily_minutes  \
0   user_99                      0              184.6   
1   user_87                      0              138.7   
2  user_870                      0               90.5   
3  user_875                      0              135.1   
4  user_573                      0               53.6   
5  user_735                      0              117.6   
6  user_757                      0              143.0   
7  user_521                      0              155.7   
8  user_200                      0              141.0   
9  user_140                      0               70.2   

   rolling_7d_intensity_avg  
0                184.600000  
1                161.650000  
2                137.933333  
3                137.225000  
4                120.500000  
5                120.016667  
6                123.300000  
7                127.350000  
8                121.900000  
9                113.337500  


In [5]:
# [STAGE 4]: Data Normalization & Persistence for Prediction Pipeline
output_path = '../data/engineered_lifecycle_features.csv' if os.path.exists('../data') else 'data/engineered_lifecycle_features.csv'
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))

df_engineered.to_csv(output_path, index=False)
print(f"Feature engineering complete. Dataset materialized at {output_path}.")

Feature engineering complete. Dataset materialized at ../data/engineered_lifecycle_features.csv.
